<a href="https://colab.research.google.com/github/ZulfiqarHusain/60-Day-AI-Challange/blob/main/Day%2017-Improve%20RAG%20Precision%20with%20Metadata%20Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages if not already installed
!pip install -qU langchain-community langchain-core faiss-cpu sentence-transformers

In [ ]:
import os
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Initialization
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Ingesting Documents explicitly WITH Metadata dictionaries
raw_documents = [
    Document(
        page_content="UrbanEye production system logs track extreme pothole counts on Bhopal road parameters.",
        metadata={"source": "engineering_team", "category": "technical", "date": "2026-05-10", "document_type": "report"}
    ),
    Document(
        page_content="UrbanEye overview pitch states that the project drastically upgrades urban smart infrastructure management.",
        metadata={"source": "marketing_team", "category": "overview", "date": "2026-01-15", "document_type": "brief"}
    ),
    Document(
        page_content="Campus Workshop data: Over 120 final-year engineering students attended the Generative AI prompt framework session.",
        metadata={"source": "campus_ambassador", "category": "academic", "date": "2025-10-22", "document_type": "summary"}
    ),
    Document(
        page_content="Archived structural road documentation from historical 2020 urban surveys across state coordinates.",
        metadata={"source": "municipal_records", "category": "historical", "date": "2020-08-19", "document_type": "archive"}
    )
]

# Build Local FAISS Vector Index with documents preserving metadata blocks
vector_store = FAISS.from_documents(raw_documents, embeddings)

# 3. Dynamic Filtered Retrieval Handler
def filtered_retrieve(query, metadata_filter=None):
    """Fetches similarity context nodes applying dynamic structural constraints directly inside FAISS search hooks."""
    # Using similarity search with score scaling or filtering dict expressions
    if metadata_filter:
        # FAISS natively supports categorical filter callable expressions inside LangChain setup
        results = vector_store.similarity_search(query, k=2, filter=metadata_filter)
    else:
        results = vector_store.similarity_search(query, k=2)

    return [doc.page_content for doc in results]

# =====================================================================
# 4. SIDE-BY-SIDE PIPELINE EVALUATION
# =====================================================================
print(f"{'QUERY & FILTER MODE':<55} | {'RETRIEVED DATA CONTEXT OVERVIEW'}")
print("="*125)

# Test 1: Category constraint (Restricts to technical report details only)
tech_filter = {"category": "technical"}
res_tech = filtered_retrieve("UrbanEye pothole system", metadata_filter=tech_filter)
print(f"{'UrbanEye system (Filter: category=technical)':<55} | {res_tech[0] if res_tech else 'No match'}")

# Test 2: Temporal exclusion cutoff check (Excluding historical data older than configurable targets)
# Simulating retrieval on same terms without older archives matching layout parameters
historical_filter = {"category": "historical"}
res_hist = filtered_retrieve("road documentation metrics", metadata_filter=historical_filter)
print(f"{'Road data survey (Filter: category=historical)':<55} | {res_hist[0] if res_hist else 'No match'}")

# Test 3: Standard Blind Retrieval (Without filter - returns first nearest neighbor conceptually)
res_blind = filtered_retrieve("UrbanEye layout parameters")
print(f"{'UrbanEye layout parameters (No Metadata Filter)':<55} | {res_blind[0] if res_blind else 'No match'}")